# 4. 寻找两个正序数组的中位数

给定两个大小分别为 m 和 n 的正序（从小到大）数组 nums1 和 nums2。请你找出并返回这两个正序数组的 中位数 。

算法的时间复杂度应该为 O(log (m+n)) 。

示例 1：
```text
输入：nums1 = [1,3], nums2 = [2]
输出：2.00000
解释：合并数组 = [1,2,3] ，中位数 2
```
示例 2：
```text
输入：nums1 = [1,2], nums2 = [3,4]
输出：2.50000
解释：合并数组 = [1,2,3,4] ，中位数 (2 + 3) / 2 = 2.5
```

## 解题思路

**中位数（Median）的物理意义**：中位数其实就是一条“分割线”（我们称之为“一刀”）。它把一个集合劈成了长度相等的两半，并且保证：左半边的最大值，永远小于等于右半边的最小值。

现在我们有两个数组 nums1 和 nums2。我们想要找的中位数，其实就是同时在两个数组上各砍一刀，把它们各自劈成左右两半，然后把两个左半边拼起来，两个右半边拼起来。

- 下刀位置 i = 0：| A B C （挑选了 0 个元素去左边
- 下刀位置 i = 1：A | B C （挑选了 1 个元素去左边
- 下刀位置 i = 2：A B | C （挑选了 2 个元素去左边
- 下刀位置 i = 3：A B C | （挑选了全部 3 个元素去左边）

为了让这“一刀”合法，必须满足两个极其严格的物理定律：

1. **数量守恒定律**：
   拼接后的左半边长度，必须等于右半边长度（如果是奇数个，左半边可以多一个）。

    假设总长度为 `len`，那么左边总共需要 `(m + n +    1) // 2` 个元素。

    这意味着，如果你在 `nums1` 的第 `i` 个位置砍一刀，那么你在 `nums2` 砍刀的位置 `j` 是**被完全锁死**的：`j = (m + n + 1) // 2 - i`。

2. **交叉小等定律（核心灵魂）**：

    切完之后，切口附近会产生四个临界值：

    - `nums1` 切口左边的数：`L1`
    - `nums1` 切口右边的数：`R1`
    - `nums2` 切口左边的数：`L2`
    - `nums2` 切口右边的数：`R2`

    因为数组本身是有序的，所以天然满足 `L1 <= R1` 和 `L2 <= R2`。

    但要保证整个左半边拼起来小于等于右半边，必须满足**交叉小于等于**：

    - **`L1 <= R2`**
    - **`L2 <= R1`**

> 神奇的统一公式：`(m + n + 1) // 2`。
> 
> 为了让奇数和偶数都能用**同一个公式**算出左半边应该分到几个元素，前人总结出了这个神仙公式：`left_total = (m + n + 1) // 2` （注意这里是向下取整）。
>
> - **当总长度是奇数时（例如 $m+n = 5$）：**  `left_total = (5 + 1) // 2 = 3`。左边分到 3 个，右边分到 2 个。**左边刚好比右边多 1 个！**既然左边有 3 个数，右边有 2 个数，那么把它们排好序后，最中间的那个数（第 3 个数）是不是肯定落在左半边？所以在奇数情况下，中位数直接就是**左半边的最大值** `max(L1, L2)`。代码写起来极其干净。
> - **当总长度是偶数时（例如 $m+n = 4$）：**`left_total = (4 + 1) // 2 = 5 // 2 = 2`。左边分到 2 个，右边分到 2 个。**左右绝对平分！**中位数就是左半边最大值和右半边最小值的平均数：`(max(L1, L2) + min(R1, R2)) / 2`。

## 二分查找的目标：在较短的数组里移动“这把刀”

只要找到了同时满足 `L1 <= R2` 且 `L2 <= R1` 的那个切割点 `i`，中位数就直接呼之欲出了！

- 如果是偶数：`max(L1, L2)` 和 `min(R1, R2)` 的平均值。
- 如果是奇数：左半边多一个，直接是 `max(L1, L2)`。

## 代码

In [ ]:
from typing import List

class Solution:
    def findMedianSortedArrays(self, nums1: List[int], nums2: List[int]) -> float:
        # 为了保证时间复杂度最优，且防止 nums2 的切割点 j 出现负数越界
        # 我们永远对较短的数组进行二分切割
        if len(nums1) > len(nums2):
            nums1, nums2 = nums2, nums1
            
        m, n = len(nums1), len(nums2)
        
        # left_total 是左半边需要的总元素个数
        left_total = (m + n + 1) // 2 
        
        # 在 nums1 中进行二分查找，寻找切割点 i
        left, right = 0, m
        
        while left <= right:
            # i 是 nums1 的切割点，j 是 nums2 的切割点
            i = (left + right) // 2
            j = left_total - i
            
            # 处理越界的极值情况：切在最左侧或最右侧
            L1 = float('-inf') if i == 0 else nums1[i - 1]
            R1 = float('inf') if i == m else nums1[i]
            L2 = float('-inf') if j == 0 else nums2[j - 1]
            R2 = float('inf') if j == n else nums2[j]
            
            # 完美的交叉校验
            if L1 <= R2 and L2 <= R1:
                # 找到了完美切割点！
                # 根据总长度奇偶性计算中位数
                if (m + n) % 2 == 1:
                    return max(L1, L2)
                else:
                    return (max(L1, L2) + min(R1, R2)) / 2.0
                    
            elif L1 > R2:
                # nums1 的左半边太大，说明刀切得太靠右了，往左收缩
                right = i - 1
            else:
                # nums2 的左半边太大 (L2 > R1)，说明 nums1 的刀切得太靠左了，往右推
                left = i + 1